# teaching logistic regression: will the mess lunch be good today?

quick walkthrough explaining how our model predicts if college mess food is edible (1) or trash (0) from 4 numbers:
- `queue length`: people in line
- `plate waste fraction`: how much got thrown in the bin
- `menu board kcal`: calories
- `handwriting neatness`: handwriting score 1-10


## 1. why linear regression doesnt work here
linear regression gives straight line values like 47 or -5.
for a probability we strictly need numbers between 0 and 1.
so we pass the linear equation through sigmoid to squash it into [0, 1].


In [ ]:
import math

def sigmoid(val):
    if val >= 0.0:
        return 1.0 / (1.0 + math.exp(-val))
    ex = math.exp(val)
    return ex / (1.0 + ex)

print("sigmoid(-10):", round(sigmoid(-10), 4))
print("sigmoid(0):  ", round(sigmoid(0), 4))
print("sigmoid(10): ", round(sigmoid(10), 4))
print("sigmoid(47): ", round(sigmoid(47), 4))


## 2. load data & scale to 0-1
kcal is ~2000 while waste is ~0.5. big numbers overpower small numbers in dot product, so we min-max scale everything to [0, 1] using min and max from train split only.


In [ ]:
import csv

def loaddata(filepath):
    rows = []
    with open(filepath, "r") as filehandle:
        reader = csv.reader(filehandle)
        header = next(reader)
        for line in reader:
            feats = [float(val) for val in line[:4]]
            target = int(line[4])
            rows.append((feats, target))
    return rows

def fitscaler(trainrows):
    mins = []
    maxs = []
    for colidx in range(4):
        colvals = [row[0][colidx] for row in trainrows]
        mins.append(min(colvals))
        maxs.append(max(colvals))
    return mins, maxs

def scaledata(rows, mins, maxs):
    normrows = []
    for row in rows:
        normrow = []
        for colidx in range(4):
            colmin = mins[colidx]
            colmax = maxs[colidx]
            if colmax == colmin:
                normrow.append(0.0)
            else:
                normrow.append((row[0][colidx] - colmin) / (colmax - colmin))
        normrows.append((normrow, row[1]))
    return normrows

import os
datapath = ""
for p in ["../../datasets/binary_classification.csv", "datasets/binary_classification.csv", "/home/sdheeraj/Documents/GitHub/Hacktoberfest26-AIML-Challenges/datasets/binary_classification.csv"]:
    if os.path.exists(p):
        datapath = p
        break
rawdata = loaddata(datapath)
trainrows = rawdata[:400]
testrows = rawdata[400:]
mins, maxs = fitscaler(trainrows)
trainnorm = scaledata(trainrows, mins, maxs)
testnorm = scaledata(testrows, mins, maxs)
print("loaded", len(trainnorm), "train rows and", len(testnorm), "test rows")


## 3. predicting probabilities
dot product of weights and features, add bias, squash with sigmoid.
if p >= 0.5 we guess good (1), else bad (0).


In [ ]:
def dotproduct(veca, vecb):
    tot = 0.0
    for i in range(len(veca)):
        tot += veca[i] * vecb[i]
    return tot

def predictproba(weights, bias, feats):
    return sigmoid(dotproduct(weights, feats) + bias)

def predictlabel(weights, bias, feats):
    p = predictproba(weights, bias, feats)
    return 1 if p >= 0.5 else 0


## 4. measuring loss (binary cross-entropy)
punishes wrong confident guesses heavily. if actual is 1 and we guess 0.99, loss is ~0. if we guess 0.01, loss blows up.


In [ ]:
def computeloss(weights, bias, dataset):
    totloss = 0.0
    n = len(dataset)
    for feats, target in dataset:
        p = predictproba(weights, bias, feats)
        pclip = min(max(p, 1e-15), 1.0 - 1e-15)
        totloss += -(target * math.log(pclip) + (1.0 - target) * math.log(1.0 - pclip))
    return totloss / n


## 5. gradient descent
compute slope for each weight and step in the opposite direction.


In [ ]:
def trainstep(weights, bias, dataset, learningrate):
    n = len(dataset)
    numfeats = len(weights)
    dw = [0.0] * numfeats
    db = 0.0
    for feats, target in dataset:
        p = predictproba(weights, bias, feats)
        diff = p - target
        for i in range(numfeats):
            dw[i] += diff * feats[i]
        db += diff
    newweights = [weights[i] - learningrate * (dw[i] / n) for i in range(numfeats)]
    newbias = bias - learningrate * (db / n)
    return newweights, newbias


## 6. train loop
start with 0 weights and run 2000 epochs at lr = 0.1.


In [ ]:
weights = [0.0] * 4
bias = 0.0
learningrate = 0.1

for epoch in range(2001):
    weights, bias = trainstep(weights, bias, trainnorm, learningrate)
    if epoch % 500 == 0:
        lossval = computeloss(weights, bias, trainnorm)
        print("epoch", epoch, "loss:", round(lossval, 4))

featnames = ["queue length", "plate waste", "calories", "neatness"]
print("\nlearned weights:")
for i in range(4):
    print(" ", featnames[i], ":", round(weights[i], 3))
print("  bias:", round(bias, 3))


## 7. testing accuracy
let us see what we get on the 100 unseen lunches.


In [ ]:
correct = 0
for feats, target in testnorm:
    pred = predictlabel(weights, bias, feats)
    if pred == target:
        correct += 1

accuracy = correct / len(testnorm)
print("test set accuracy:", round(accuracy * 100, 1), "%")
